# Аналитика продаж в amoCRM

Что происходит по шагам:
1. Скрипт подтягивает из amoCRM новые и изменённые сделки и обновляет таблицу в PostgreSQL
2. Выполняется ряд SQL-запросов с аналитическими метриками.
3. Результаты выводятся ниже




In [ ]:
# ========== НАСТРОЙКИ ==========

AMO_SUBDOMAIN = ""        
AMO_TOKEN = ""   

PG_CONNECTION = ""

TABLE_NAME = ""  

In [ ]:
# ========== ШАГ 1. СИНХРОНИЗАЦИЯ: amoCRM -> PostgreSQL ==========
# Логика: смотрим самую свежую дату updated_at в базе,
# качаем из amoCRM только то, что изменилось после неё,
# старые версии этих записей удаляем, свежие дописываем.

import time
import requests
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(PG_CONNECTION)
headers = {"Authorization": "Bearer " + AMO_TOKEN}


def get_leads_from_amo(updated_from=None):
    """Скачивает сделки из amoCRM. Если передать updated_from (unix-время),
    скачает только те, что изменились после этого времени."""
    url = f"https://{AMO_SUBDOMAIN}.amocrm.ru/api/v4/leads"

    all_leads = []
    page = 1

    while True:
        params = {"limit": 250, "page": page}
        if updated_from is not None:
            params["filter[updated_at][from]"] = updated_from

        response = requests.get(url, headers=headers, params=params)

        # 204 значит "данных нет" — выходим
        if response.status_code == 204:
            break

        # если amoCRM вернул ошибку (например 401 — неверный токен),
        # останавливаемся с понятным сообщением
        response.raise_for_status()

        data = response.json()
        leads = data["_embedded"]["leads"]
        all_leads = all_leads + leads

        # нет ссылки на следующую страницу — это была последняя
        if "next" not in data["_links"]:
            break

        page = page + 1
        time.sleep(0.2)  # чтобы не упереться в лимиты amoCRM

    return all_leads


def make_dataframe(leads):
    """Берём из каждой сделки только нужные поля."""
    rows = []
    for lead in leads:
        rows.append({
            "id": lead["id"],
            "name": lead.get("name"),
            "price": lead.get("price"),
            "status_id": lead.get("status_id"),
            "pipeline_id": lead.get("pipeline_id"),
            "responsible_user_id": lead.get("responsible_user_id"),
            "created_at": lead.get("created_at"),
            "updated_at": lead.get("updated_at"),
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], unit="s")
        df["updated_at"] = pd.to_datetime(df["updated_at"], unit="s")

    return df


def get_last_update_time():
    """Самый свежий updated_at из таблицы (unix-время).
    Если таблицы ещё нет — None."""
    with engine.connect() as conn:
        table_exists = conn.execute(
            text("SELECT to_regclass(:name)"), {"name": TABLE_NAME}
        ).scalar()

        if table_exists is None:
            return None

        result = conn.execute(
            text(f"SELECT EXTRACT(EPOCH FROM MAX(updated_at)) FROM {TABLE_NAME}")
        ).scalar()

        if result is None:
            return None
        return int(result)


def save_to_postgres(df):
    """Удаляем строки с такими же id и дописываем свежие версии."""
    if df.empty:
        print("Новых записей нет.")
        return

    ids = df["id"].tolist()

    # удаление старых версий и вставка свежих — в одной транзакции:
    # либо выполнится всё вместе, либо ничего
    with engine.begin() as conn:
        table_exists = conn.execute(
            text("SELECT to_regclass(:name)"), {"name": TABLE_NAME}
        ).scalar()

        if table_exists is not None:
            conn.execute(
                text(f"DELETE FROM {TABLE_NAME} WHERE id = ANY(:ids)"),
                {"ids": ids},
            )

        df.to_sql(TABLE_NAME, conn, index=False, if_exists="append")

    print(f"Записано в базу: {len(df)} строк")


def sync():
    last_time = get_last_update_time()

    if last_time is None:
        print("Первая загрузка — качаем всё.")
        leads = get_leads_from_amo()
    else:
        print("Качаем только изменения.")
        leads = get_leads_from_amo(updated_from=last_time + 1)

    df = make_dataframe(leads)
    print(f"Получено из amoCRM: {len(df)} записей")

    save_to_postgres(df)
    return df


fresh_df = sync()

In [ ]:
# ========== ШАГ 2. SQL-ЗАПРОСЫ С МЕТРИКАМИ ==========
# Важно: в amoCRM у закрытых сделок специальные статусы:
#   142 — "Успешно реализовано" (выиграна)
#   143 — "Закрыто и не реализовано" (проиграна)
# Всё остальное — сделки в работе.
#
# МЕТРИКИ:
#   1. Воронка по деньгам (этапы 3-7) — где лежат потенциальные деньги
#   2. Конверсия по месяцам — % закрытых сделок, улучшается ли тренд
#   3. Выручка по месяцам — доход от выигранных сделок
#   4. Новые заявки по неделям — входящий поток лидов (стабильность)
#   5. Зависшие сделки (14+ дней) — не трогали, нужна работа менеджера
#   6. Среднее время закрытия — как быстро мы закрываем сделку
#   7. Дни в каждом этапе — на каком этапе bottleneck
#   8. Средний чек по этапам — какой размер сделок на каждом этапе
#   9. Win Rate по менеджерам — % закрытий у каждого
#  10. Выручка по менеджерам — кто сколько денег принёс
#  11. Дни без движения — для каждой сделки: когда последний раз её трогали
#  12. Средний чек: выигранные vs проигранные — разница в размере
#  13. Старые активные сделки (60+ дней) — замёрзшие, нужна переквалификация


queries = {

    "1. Воронка: сделки и деньги в работе по этапам": """
        SELECT status_id,
               COUNT(*)   AS deals,
               SUM(price) AS total_sum
        FROM amo_leads
        WHERE status_id NOT IN (142, 143)
          AND status_id >= 3
          AND price IS NOT NULL
          AND price > 0
        GROUP BY status_id
        ORDER BY total_sum DESC NULLS LAST
    """,

    "2. Win Rate по месяцам (выиграно / проиграно)": """
        SELECT DATE_TRUNC('month', updated_at)::date AS month,
               COUNT(*) FILTER (WHERE status_id = 142) AS won,
               COUNT(*) FILTER (WHERE status_id = 143) AS lost,
               ROUND(100.0 * COUNT(*) FILTER (WHERE status_id = 142) /
                     NULLIF(COUNT(*), 0)) AS conversion_pct
        FROM amo_leads
        WHERE status_id IN (142, 143)
        GROUP BY month
        ORDER BY month
    """,

    "3. Выручка по выигранным сделкам по месяцам": """
        SELECT DATE_TRUNC('month', updated_at)::date AS month,
               COUNT(*)   AS won_deals,
               SUM(price) AS revenue
        FROM amo_leads
        WHERE status_id = 142
        GROUP BY month
        ORDER BY month
    """,
    # Для закрытых сделок updated_at используется как дата последнего изменения
    # записи и приближённо отражает дату закрытия.

    "4. Новые заявки по неделям (за 3 месяца)": """
        SELECT DATE_TRUNC('week', created_at)::date AS week,
               COUNT(*) AS new_leads
        FROM amo_leads
        WHERE created_at >= CURRENT_DATE - INTERVAL '3 months'
        GROUP BY week
        ORDER BY week
    """,

    "5. Зависшие сделки (не трогали 14+ дней, но не старше 365)": """
    SELECT id, name, price, status_id, responsible_user_id,
           updated_at::date AS last_touch,
           CURRENT_DATE - updated_at::date AS days_without_touch
    FROM amo_leads
    WHERE status_id NOT IN (142, 143)
      AND updated_at < CURRENT_DATE - INTERVAL '14 days'
      AND updated_at >= CURRENT_DATE - INTERVAL '365 days'
    ORDER BY price DESC NULLS LAST
    LIMIT 100
    """,

    "6. Среднее время закрытия (за последние 90 дней)": """
    SELECT 
        ROUND(AVG(EXTRACT(DAY FROM updated_at - created_at))) AS avg_days_to_close,
        MIN(EXTRACT(DAY FROM updated_at - created_at)) AS min_days,
        MAX(EXTRACT(DAY FROM updated_at - created_at)) AS max_days
    FROM amo_leads
    WHERE status_id IN (142, 143)
      AND created_at >= CURRENT_DATE - INTERVAL '90 days'
    """,

    "7. Средний возраст активных сделок по этапам": """
        SELECT 
            status_id,
            COUNT(*) as deals_count,
            ROUND(AVG(CURRENT_DATE - created_at::date)) AS avg_days_in_stage
        FROM amo_leads
        WHERE status_id NOT IN (142, 143)
        AND status_id IN (81775006, 81775010, 81775014, 81775038, 85181238, 
                        85181090, 85181094, 81775042, 81775046, 81775050)
          AND created_at >= CURRENT_DATE - INTERVAL '180 days'
        GROUP BY status_id
        ORDER BY status_id
    """,

    "8. Среднее значение сделки по этапам": """
        SELECT 
            status_id,
            COUNT(*) AS deals,
            ROUND(AVG(price)) AS avg_deal_size,
            SUM(price) AS total_sum
        FROM amo_leads
        WHERE status_id NOT IN (142, 143)
          AND price IS NOT NULL
        GROUP BY status_id
        ORDER BY status_id
    """,

    "9. Win Rate по менеджерам": """
        SELECT 
            responsible_user_id,
            COUNT(*) FILTER (WHERE status_id = 142) AS won,
            COUNT(*) FILTER (WHERE status_id = 143) AS lost,
            ROUND(100.0 * COUNT(*) FILTER (WHERE status_id = 142) / 
                  NULLIF(COUNT(*), 0), 1) AS win_rate_pct
        FROM amo_leads
        WHERE status_id IN (142, 143)
        GROUP BY responsible_user_id
        ORDER BY win_rate_pct DESC
    """,

    "10. Выручка по менеджерам": """
        SELECT 
            responsible_user_id,
            COUNT(*) AS won_deals,
            SUM(price) AS revenue,
            ROUND(AVG(price)) AS avg_deal_size
        FROM amo_leads
        WHERE status_id = 142
        GROUP BY responsible_user_id
        ORDER BY revenue DESC
    """,

    "11. Сколько дней сидит сделка БЕЗ движения": """
        SELECT 
            id, name, price, status_id, responsible_user_id,
            CURRENT_DATE - updated_at::date AS days_without_touch
        FROM amo_leads
        WHERE status_id NOT IN (142, 143)
        ORDER BY days_without_touch DESC
    """,

    "12. Средний чек: выигранные vs проигранные": """
        SELECT 
            CASE 
                WHEN status_id = 142 THEN 'Выигранные'
                WHEN status_id = 143 THEN 'Проигранные'
            END AS result,
            COUNT(*) AS deals,
            ROUND(AVG(price)) AS avg_price,
            SUM(price) AS total_revenue
        FROM amo_leads
        WHERE status_id IN (142, 143)
        GROUP BY result
    """,

    "13. Самые старые активные сделки (60+ дней, топ 50)": """
    SELECT 
        id, name, price, status_id, responsible_user_id,
        created_at::date AS created,
        updated_at::date AS last_update,
        CURRENT_DATE - created_at::date AS days_in_system
    FROM amo_leads
    WHERE status_id NOT IN (142, 143)
      AND created_at < CURRENT_DATE - INTERVAL '60 days'
    ORDER BY price DESC NULLS LAST
    LIMIT 50
""",

}

In [ ]:
# ========== ШАГ 3. СЧИТАЕМ, ВЫВОДИМ И СОХРАНЯЕМ МЕТРИКИ ==========
# Каждая метрика после расчёта дописывается в свою таблицу в Postgres
# (metrics_history_1..5) с датой расчёта — так копится история,
# и через месяц можно смотреть, как менялись цифры.

import datetime

today = datetime.date.today()
results = {}

for i, (name, sql) in enumerate(queries.items(), start=1):
    df_m = pd.read_sql(sql, engine)
    results[name] = df_m

    # показываем на экране
    print("=" * 50)
    print(name)
    display(df_m)

    # сохраняем в историю: добавляем дату расчёта и дописываем в таблицу
    df_m["calc_date"] = today
    df_m.to_sql(f"metrics_history_{i}", engine, index=False, if_exists="append")

print("=" * 50)
print("Готово! Метрики посчитаны и записаны в историю.")